In [15]:
import json
import requests
from openai import OpenAI

client = OpenAI()


def get_popular_movies():
    url = "https://nomad-movies.nomadcoders.workers.dev/movies"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_movie_details(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_movie_credits(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/credits"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()

def get_movie_videos(id):
    # get movide videos by id   
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/videos"
    resp = requests.get(url, timeout= 10)
    resp.raise_for_status()
    return resp.json()


TOOLS = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Get a list of currently popular movies.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Get detailed information about a movie by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The movie id"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_credits",
        "description": "Get cast and crew credits for a movie by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The movie id"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
    {
        "type":"function",
        "name":"get_movie_videos",
        "description": "Get videos that is related with movie by id",
        "parameters": {
            "type":"object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "the movie id"
                },
            },
            "required":["id"],
            "additionalProperties": False
        }
    }
]


def call_tool(tool_name, args):
    if tool_name == "get_popular_movies":
        return get_popular_movies()
    elif tool_name == "get_movie_details":
        return get_movie_details(args["id"])
    elif tool_name == "get_movie_credits":
        return get_movie_credits(args["id"])
    elif tool_name == "get_movie_videos":
        return get_movie_videos(args["id"])
    else:
        raise ValueError(f"Unknown tool: {tool_name}")


def run_agent(user_input):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=user_input,
        instructions=("You are a helpful movie assistant. "
        "Use the provided tools whenever the user asks for real movie data. " 
        "If the user asks for popular movies, use get_popular_movies and include each movie id next to the title." 
        "ID is REQUIRED NEXT TO TITLE ! "
        "If the user asks for details, use get_movie_details. " 
        "If the user asks for cast or crew, use get_movie_credits. "
        "If the user asks for trailers or videos, use get_movie_videos. "
        "Answer clearly in Korean."
        ),
        tools=TOOLS,
    )

    function_calls = [
        item for item in response.output
        if item.type == "function_call"
    ]

    if not function_calls:
        return response.output_text

    call = function_calls[0]

    tool_name = call.name
    args = json.loads(call.arguments)

    print(f"🔧 Function Call:, {tool_name}, and the ID is {args}")

    tool_result = call_tool(tool_name, args)

    final_response = client.responses.create(
        model="gpt-4o-mini",
        previous_response_id=response.id,
        instructions="If the user asks for popular movies, use get_popular_movies and include each movie id next to the title." 
        "ID is REQUIRED NEXT TO TITLE ! ",
        input=[{
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(tool_result, ensure_ascii=False),
        }],
    )

    return final_response.output_text


In [16]:

result = run_agent("지금 인기 있는 영화 2개만 알려줘")
print(result)


🔧 Function Call:, get_popular_movies, and the ID is {}
현재 인기 있는 영화 두 편은 다음과 같습니다:

1. **Your Heart Will Be Broken** (ID: 1523145)  
   ![Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)  
   고등학생 폴리나가 새로운 학교에서 괴롭힘을 당할 때, 주동적 괴롭힘자인 바르스와 계약을 맺고 서로를 보호하기 위한 연인 역할을 하게 되면서 진짜 감정이 생기게 됩니다.

2. **Avatar: Fire and Ash** (ID: 83533)  
   ![Avatar: Fire and Ash](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)  
   제이크 설리와 네이티리 부부는 판도라에서 새로운 위협인 아쉬 사람들과 맞서 싸우며, 가족을 위한 생존을 위해 고군분투합니다.


In [17]:

result = run_agent("ID: 1523145  대해서 더알려줘")
print(result)


🔧 Function Call:, get_movie_details, and the ID is {'id': 1523145}
### 영화 정보: **Your Heart Will Be Broken** (ID: 1523145)

- **원제**: Твоё сердце будет разбито
- **장르**: 드라마, 로맨스
- **상영 시간**: 134 분
- **평점**: 6.9 (55표)
- **개봉일**: 2026-03-26
- **상태**: 개봉됨
- **언어**: 러시아어, 스페인어

#### 줄거리
고등학생 폴리나가 새로운 학교에서 괴롭힘을 당하는 중, 주동자인 바르스와 거래를 하게 된다. 그는 그녀의 남자친구인 척하며 그녀를 지켜봐야 하고, 폴리나는 그가 시키는 모든 것을 해야 한다. 이 게임을 통해 두 사람은 진정한 감정을 느끼게 되지만, 그녀의 가족과 급우들은 이 커플을 갈라놓으려 한다.

#### 제작사
- All Media A Start Company
- START Studio
- Sverdlovsk Film Studio
- Cinema Foundation of Russia
- Yellow, Black & White
- Premier Studios

#### 포스터
![Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)

추가적인 질문이 있으시면 언제든지 말씀해 주세요!


In [18]:

result = run_agent("movie ID: 1523145 에 예고편이나 그런 영상있니?")
print(result)


🔧 Function Call:, get_movie_videos, and the ID is {'id': 1523145}
현재 영화 ID 1523145에 대한 예고편이나 관련 영상이 없습니다. 다른 정보가 필요하시면 말씀해 주세요!
